# MS-MARCO MiniLM-L12 Cross-Encoder Reranker - AWS Marketplace

Deploys **MS-MARCO MiniLM-L12 Cross-Encoder Reranker** from AWS Marketplace as a SageMaker endpoint inside **your own AWS account**. Your data never leaves your VPC and there are no external API calls or token limits.

Cross-encoder reranker based on MiniLM-L12 fine-tuned on MS-MARCO. Larger than L6 variant with improved ranking accuracy.

## Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # recommended real-time instance
ENDPOINT_NAME = "ms-marco-minilm-l12-reranker"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills per hour while it exists, so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Rerank passages

The endpoint accepts `application/json` shaped `{"query": "...", "passages": ["...", "..."]}`.
Returns `{"scores": [0.94, 0.89]}` — one relevance score per passage, higher = more relevant.

In [ ]:
runtime = boto3.client("sagemaker-runtime")


def rerank(query, passages):
    """Return relevance scores for each (query, passage) pair."""
    payload = {"query": query, "passages": passages}
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())


query = "what is machine learning?"
passages = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Python is a programming language.",
]

result = rerank(query, passages)
print("scores:", result["scores"])

# Re-rank passages by score
ranked = sorted(zip(result["scores"], passages), reverse=True)
for score, passage in ranked:
    print(f"{score:.4f}  {passage}")

## 4. Batch transform for offline workloads

For processing large datasets without a live endpoint, batch transform avoids paying for an always-on endpoint. Input is JSON Lines, one `{"inputs": "..."}` object per line.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type=INSTANCE_TYPE,
#     output_path=f"s3://{session.default_bucket()}/ms-marco-minilm-l12-reranker/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/ms-marco-minilm-l12-reranker-input/",
#     content_type="application/json",
# )
# transformer.wait()

## 5. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)